# VoiceOps Studio — batch Colab launcher

This notebook is a thin launcher. The canonical workflow source of truth is `elevenlabs_api.py` in this repository.

Runtime files are pinned to a reviewed commit and checked by SHA-256 before execution. To update, review the new commit and update both GITHUB_REF and the expected hashes in this launcher.


In [ ]:
import pathlib
import hashlib
import subprocess
import urllib.request

GITHUB_OWNER = "Just9120"
GITHUB_REPO = "voiceops-studio"
GITHUB_REF = "613f20be225868a5170d638ec1be0d3130d0385f"  # reviewed immutable release

if len(GITHUB_REF) != 40 or any(c not in "0123456789abcdef" for c in GITHUB_REF):
    raise RuntimeError("A reviewed full commit SHA is required for Colab runtime.")

RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_REF}"
REQ_URL = f"{RAW_BASE}/requirements-colab.txt"
SRC_URL = f"{RAW_BASE}/elevenlabs_api.py"

print(f"Using GitHub ref: {GITHUB_REF}")
print(f"requirements-colab.txt URL: {REQ_URL}")
print(f"elevenlabs_api.py URL: {SRC_URL}")

workdir = pathlib.Path('/content')
req_path = workdir / 'requirements-colab.txt'
src_path = workdir / 'elevenlabs_api.py'

def download_verified(url, destination, expected_sha256):
    with urllib.request.urlopen(url, timeout=30) as response:
        payload = response.read(2 * 1024 * 1024 + 1)
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise RuntimeError("Downloaded Colab runtime does not match the reviewed release; nothing will execute.")
    destination.write_bytes(payload)

download_verified(REQ_URL, req_path, "8a3f942e3ef09c01dd821ede9c972505b16a3306c300b21528c3a9498a2eebe1")
download_verified(SRC_URL, src_path, "60cff939888b3b72b6ee62289104652a9adbbe865df5bb4138f51270c66aa6c6")

subprocess.run(['python', '-m', 'pip', 'install', '-r', str(req_path)], check=True)

source = src_path.read_text(encoding='utf-8')
lines = source.splitlines()
sanitized_lines = []
removed_known_pip_line = False
expected_pip_markers = (
    'google-api-python-client',
    'google-auth-httplib2',
    'google-auth-oauthlib',
    'requests',
    'ipywidgets',
)
for idx, line in enumerate(lines, start=1):
    stripped = line.lstrip()
    if stripped.startswith('!') or stripped.startswith('%'):
        is_expected_top_level_pip = (
            stripped.startswith('!pip ')
            and idx <= 20
            and any(marker in stripped for marker in expected_pip_markers)
        )
        if is_expected_top_level_pip and not removed_known_pip_line:
            removed_known_pip_line = True
            print(f"Removed known Colab dependency install line at {idx}: {stripped}")
            continue
        if stripped.startswith('!pip '):
            raise RuntimeError(
                f"Unexpected !pip line in elevenlabs_api.py at line {idx}: {line!r}. \
Only the known top-level Colab dependency install line is allowed."
            )
        raise RuntimeError(
            f"Unsupported notebook magic/shell line in elevenlabs_api.py at line {idx}: {line!r}. \
Refuse to guess. Please sanitize upstream file or run directly in Colab."
        )
    sanitized_lines.append(line)

exec_globals = {'__name__': '__main__', '__file__': str(src_path)}
exec(compile('\n'.join(sanitized_lines) + '\n', str(src_path), 'exec'), exec_globals)
